# VGG16 + SVM for Brain Tumor MRI Classification

This notebook uses a pretrained VGG16 backbone as a frozen feature extractor, then trains a classical SVM classifier on the extracted MRI features.

In [1]:
%pip install torch torchvision scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import joblib
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from torchvision.models import VGG16_Weights

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
project_root = Path.cwd().resolve()
if project_root.name == "Notebooks":
    project_root = project_root.parent

data_root = project_root / "Dataset" / "BT-MRI Dataset" / "BT-MRI Dataset"
challenging_root = project_root / "Dataset" / "Challenging Datasets" / "Challenging Datasets"
model_output_path = project_root / "models" / "vgg16_svm.joblib"
model_output_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Using device: {device}")
print(f"Project root: {project_root}")
print(f"Dataset root: {data_root}")

if not data_root.exists():
    raise FileNotFoundError(f"Dataset not found at {data_root}")


Using device: cuda
Project root: /media/tst_imperial/Projects/Brain_tumor/Brain_tu
Dataset root: /media/tst_imperial/Projects/Brain_tumor/Brain_tu/Dataset/BT-MRI Dataset/BT-MRI Dataset


In [3]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

feature_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

train_dataset = datasets.ImageFolder(data_root / "Training", transform=feature_transform)
test_dataset = datasets.ImageFolder(data_root / "Testing", transform=feature_transform)

loader_kwargs = {"batch_size": 32, "shuffle": False, "num_workers": 2}
if torch.cuda.is_available():
    loader_kwargs["pin_memory"] = True

train_loader = DataLoader(train_dataset, **loader_kwargs)
test_loader = DataLoader(test_dataset, **loader_kwargs)

class_names = train_dataset.classes
print(f"Classes: {class_names}")
print(f"Training images: {len(train_dataset)}")
print(f"Testing images: {len(test_dataset)}")


Classes: ['Glioma', 'Meningioma', 'No-tumor', 'Pituitary']
Training images: 5712
Testing images: 1311


In [4]:
vgg16 = models.vgg16(weights=VGG16_Weights.DEFAULT).to(device)
vgg16.eval()
global_pool = nn.AdaptiveAvgPool2d((1, 1)).to(device)

def extract_features(dataloader):
    feature_batches = []
    label_batches = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=torch.cuda.is_available())
            features = vgg16.features(images)
            features = global_pool(features)
            features = torch.flatten(features, 1)

            feature_batches.append(features.cpu().numpy())
            label_batches.append(labels.numpy())

    return np.concatenate(feature_batches), np.concatenate(label_batches)

X_train, y_train = extract_features(train_loader)
X_test, y_test = extract_features(test_loader)

print(f"Training feature matrix: {X_train.shape}")
print(f"Testing feature matrix: {X_test.shape}")


Training feature matrix: (5712, 512)
Testing feature matrix: (1311, 512)


In [5]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    (
        "svm",
        LinearSVC(
            C=1.0,
            class_weight="balanced",
            dual="auto",
            max_iter=5000,
            random_state=SEED,
        ),
    ),
])

svm_pipeline.fit(X_train, y_train)

joblib.dump(
    {
        "model": svm_pipeline,
        "class_names": class_names,
        "feature_shape": X_train.shape[1],
        "backbone": "VGG16 features + adaptive average pooling",
    },
    model_output_path,
)

print(f"Saved artifact to {model_output_path}")


Saved artifact to /media/tst_imperial/Projects/Brain_tumor/Brain_tu/models/vgg16_svm.joblib


In [6]:
y_pred = svm_pipeline.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=class_names))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


Test accuracy: 90.47%

Classification report:
              precision    recall  f1-score   support

      Glioma       0.91      0.84      0.87       300
  Meningioma       0.81      0.83      0.82       306
    No-tumor       0.97      1.00      0.98       405
   Pituitary       0.92      0.92      0.92       300

    accuracy                           0.90      1311
   macro avg       0.90      0.90      0.90      1311
weighted avg       0.90      0.90      0.90      1311

Confusion matrix:
[[251  41   3   5]
 [ 24 255   9  18]
 [  0   0 403   2]
 [  2  19   2 277]]


In [7]:
challenging_folders = [
    "Blurred Dataset",
    "Noisy Dataset",
    "Patient Motion Artifact Dataset",
]

def evaluate_folder(folder_name):
    folder_path = challenging_root / folder_name
    if not folder_path.exists():
        print(f"{folder_name}: skipped because {folder_path} was not found")
        return

    dataset = datasets.ImageFolder(folder_path, transform=feature_transform)
    dataloader = DataLoader(dataset, **loader_kwargs)
    X_eval, y_eval = extract_features(dataloader)
    y_pred = svm_pipeline.predict(X_eval)
    accuracy = accuracy_score(y_eval, y_pred)
    print(f"{folder_name}: {accuracy * 100:.2f}% accuracy across {len(dataset)} images")

for folder in challenging_folders:
    evaluate_folder(folder)


Blurred Dataset: 79.01% accuracy across 1677 images
Noisy Dataset: 56.23% accuracy across 1677 images
Patient Motion Artifact Dataset: 48.48% accuracy across 559 images
